In [ ]:
from huggingface_hub import HfFileSystem

repo_id = "nexar-ai/nexar_collision_prediction" 

fs = HfFileSystem()
root_path = f"datasets/{repo_id}"

for path, dirs, files in fs.walk(root_path):
    print(f"\n {path}")
    print(f"Folders ({len(dirs)}): {dirs[:10]}")
    print(f"Files ({len(files)}): {files[:10]}")

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
from datasets import load_dataset

repo_id = "nexar-ai/nexar_collision_prediction"

try:
    solution_path = hf_hub_download(repo_id=repo_id, filename="solution.csv", repo_type="dataset")
    df_solution = pd.read_csv(solution_path)
    print(" 5 سطر اول solution.csv:")
    print(df_solution.head())
    print("\nستون‌های solution.csv:", df_solution.columns.tolist())
except Exception as e:
    print("خطا در دریافت solution.csv:", e)

print("\n" + "="*50 + "\n")

try:
    map_path = hf_hub_download(repo_id=repo_id, filename="time_to_accident_test_map.csv", repo_type="dataset")
    df_map = pd.read_csv(map_path)
    print(" 5 سطر اول time_to_accident_test_map.csv:")
    print(df_map.head())
    print("\nستون‌های time_to_accident_test_map.csv:", df_map.columns.tolist())
except Exception as e:
    print("خطا در دریافت time_to_accident_test_map.csv:", e)

print("\n" + "="*50 + "\n")

ds = load_dataset(repo_id, streaming=True, split="train")
print(" ویژگی‌های ثبت‌شده در Hugging Face (Features):")
print(ds.features)

In [ ]:
from datasets import load_dataset

ds = load_dataset("nexar-ai/nexar_collision_prediction", streaming=True, split="train")

sample = next(iter(ds))

print(" کلیدهای موجود در هر نمونه:")
print(sample.keys())

print("\n نمونه متادیتا:")
print(f"Time of Event: {sample.get('time_of_event')}")
print(f"Time of Alert: {sample.get('time_of_alert')}")
print(f"Weather: {sample.get('weather')}")
print(f"Light: {sample.get('light_conditions')}")
print(f"Scene: {sample.get('scene')}")

In [ ]:
video_data = sample['video']

print("تایپ داده ویدیو:", type(video_data))

if isinstance(video_data, list):
    print("تعداد فریم‌ها:", len(video_data))
    print("تایپ اولین فریم:", type(video_data[0]))
elif hasattr(video_data, 'shape'):
    print("ابعاد (Shape) ویدیو:", video_data.shape)
elif isinstance(video_data, dict):
    print("کلیدهای دیکشنری ویدیو:", video_data.keys())

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

os.environ["HF_TOKEN"] = hf_token

In [ ]:
import os
import gc
import json
import logging
import torch
import torch.nn.functional as F
import numpy as np
from torchvision import models
from datasets import load_dataset

def is_valid_num(val):
    if val is None:
        return False
    try:
        return not np.isnan(float(val))
    except (ValueError, TypeError):
        return False

logging.basicConfig(
    filename="extraction.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

os.makedirs("extracted_features/1", exist_ok=True)
os.makedirs("extracted_features/0", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" پردازش روی دستگاه: {device}", flush=True)
logging.info(f"Started extraction process on device: {device}")

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device).eval()

imagenet_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

ds = load_dataset("nexar-ai/nexar_collision_prediction", streaming=True, split="train")
WINDOW_SEC = 3.0

print(" شروع فرآیند استخراج ویژگی‌ها...", flush=True)

for idx, sample in enumerate(ds):
    t_event = sample.get('time_of_event')
    t_alert = sample.get('time_of_alert')
    
    is_positive = is_valid_num(t_event)
    label = 1 if is_positive else 0
    file_id = f"{idx:05d}"
    
    npy_path = f"extracted_features/{label}/{file_id}.npy"
    
    if os.path.exists(npy_path):
        continue

    try:
        decoder = sample['video']
        weather = sample.get('weather')
        light = sample.get('light_conditions')
        scene = sample.get('scene')

        try:
            fps = float(decoder.metadata.average_fps)
            total_frames = decoder.metadata.num_frames
            duration = float(decoder.metadata.duration_seconds)
        except Exception:
            fps, total_frames = 30.0, len(decoder)
            duration = total_frames / fps

        if is_positive and is_valid_num(t_alert):
            end_sec = float(t_alert)
            start_sec = max(0.0, end_sec - WINDOW_SEC)
        else:
            start_sec = max(0.0, (duration / 2.0) - (WINDOW_SEC / 2.0))
            end_sec = min(duration, start_sec + WINDOW_SEC)

        start_frame = int(start_sec * fps)
        end_frame = min(total_frames, int(end_sec * fps))

        if end_frame <= start_frame:
            logging.warning(f"Sample {idx}: invalid frame indices ({start_frame} to {end_frame})")
            continue

        frame_batch = decoder[start_frame:end_frame]
        frames_tensor = frame_batch.data if hasattr(frame_batch, 'data') else frame_batch
        frames_tensor = frames_tensor.to(device).float() / 255.0

        frames_resized = F.interpolate(frames_tensor, size=(224, 224), mode='bilinear', align_corners=False)
        frames_normalized = (frames_resized - imagenet_mean) / imagenet_std

        with torch.no_grad():
            features = resnet(frames_normalized).cpu().numpy()

        np.save(npy_path, features)
        
        meta_data = {
            "id": file_id,
            "label": label,
            "window_start_sec": start_sec,
            "window_end_sec": end_sec,
            "time_of_alert": float(t_alert) if is_valid_num(t_alert) else None,
            "time_of_event": float(t_event) if is_valid_num(t_event) else None,
            "weather": weather,
            "light_conditions": light,
            "scene": scene,
            "extracted_frames": len(features)
        }

        with open(f"extracted_features/{label}/{file_id}.json", "w", encoding="utf-8") as f:
            json.dump(meta_data, f, indent=4, ensure_ascii=False)

        if (idx + 1) % 10 == 0:
            print(f" {idx + 1} نمونه پردازش شد.", flush=True)
            logging.info(f"Successfully processed {idx + 1} samples.")

    except Exception as e:
        error_msg = f" Error on sample {idx}: {str(e)}"
        print(error_msg, flush=True)
        logging.error(error_msg)

    finally:
        if 'frames_tensor' in locals(): del frames_tensor
        if 'frames_resized' in locals(): del frames_resized
        if 'frames_normalized' in locals(): del frames_normalized
        if 'features' in locals(): del features
        
        if idx % 10 == 0:
            torch.cuda.empty_cache()
            gc.collect()

print(" پردازش تمام نمونه‌ها به پایان رسید!", flush=True)